# FreshCart ELT workflow notebook

The executable cells below are preserved as the project workflow. They have empty native outputs because this notebook was not executed for this submission update. The Markdown evidence section at the end reproduces previously captured results from the cited reports; it is not a fresh notebook execution. Do not run the Snowflake cells without an authorized execution plan.

## Load project code and connect

Install dependencies from the repository root with `python -m pip install -r requirements.txt`. The notebook reads credentials through environment variables (and loads a local `.env` if `python-dotenv` is installed). It does not display credential values.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'elt').is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from elt.freshcart_pipeline import (
    baseline_snapshot, connect_snowflake, quality_checks, query, run_pipeline,
    run_sql, setup_sql, show,
)

database = os.getenv('SNOWFLAKE_DATABASE', 'FRESHCART')
connection = connect_snowflake()
run_sql(connection, setup_sql(database))
print('Snowflake setup completed for the selected database.')

## B1. Baseline run

For the clean baseline, use a new/empty database in `SNOWFLAKE_DATABASE` before running setup. The expected one-run checks from the assignment are 8 raw orders, 6 clean rows, 2 quarantined orders, 4 deduplicated/gold orders, INR 7,785.00 revenue, 126 loyalty points, and 25% rejected. The following cells display actual Snowflake results; they do not hardcode or simulate those results.

In [ ]:
baseline_run_id = run_pipeline(connection)
baseline = baseline_snapshot(connection)
baseline

In [ ]:
show(connection, "SELECT source, COUNT(*) AS raw_rows FROM bronze.orders_raw GROUP BY source ORDER BY source")

## B2. Record traces

The two APIs use different field names and shapes. `SQL_10_ORDERS_CLEAN` maps web `order_id`, `cust_email`, `order_date`, `amount`, etc. and mobile `orderId`, `customer.email`, `orderDate`, `total.value`, etc. into the same typed columns. `SQL_20_ORDERS_DEDUP` partitions on `order_id` and orders by `updated_at DESC, ingest_seq DESC`, so the later mobile ORD-1003 record with amount 2,100 INR survives.

For ORD-1003, the expected calculation from the supplied data is `amount_inr = 2,100 × 1.0 = 2,100`; Meera's Platinum multiplier is 3, so `floor(2,100 / 100) × 3 = 63` points. This is arithmetic from the sample inputs, not a Snowflake execution result.

For Priya, `dim.customers_scd2` is joined by matching email and `order_date BETWEEN valid_from AND valid_to`. ORD-1001 (Sept 1) uses Silver: `floor(1,250 / 100) × 1 = 12`. ORD-1005 (Sept 3, the Gold effective date) uses Gold: `floor(899 / 100) × 2 = 16`.

In [ ]:
show(connection, "SELECT source, raw, run_id, extracted_at FROM bronze.orders_raw WHERE raw:order_id::STRING = 'ORD-1003' OR raw:orderId::STRING = 'ORD-1003' ORDER BY ingest_seq")

In [ ]:
show(connection, "SELECT order_id, source, order_date, amount, currency, updated_at, reasons FROM silver.orders_checked WHERE order_id = 'ORD-1003' ORDER BY updated_at")

In [ ]:
show(connection, "SELECT order_id, order_date, tier, multiplier, amount, currency, fx_rate, amount_inr, loyalty_points FROM gold.fct_orders WHERE order_id IN ('ORD-1001', 'ORD-1003', 'ORD-1005') ORDER BY order_id")

## B3. Unsupported-currency validation

The working SQL adds `unsupported_currency` for values other than INR/USD. Before the rule, ORD-1006 is rejected for `bad_email` and `bad_pincode`; after the rule, it should also include `unsupported_currency` because its currency is EUR. The order remains quarantined, so the gold total should not change. The quarantine `NOT EXISTS` check keys on source, order ID, and updated timestamp, preventing a second copy of the same rejected version. The query below displays the actual reasons after you rerun the pipeline.

In [ ]:
run_pipeline(connection)
show(connection, "SELECT source, order_id, reasons FROM quarantine.orders WHERE order_id = 'ORD-1006' ORDER BY quarantined_at")

## B4. Arjun's SCD Type 2 change

Prediction before execution: CUST-02 has 2 SCD2 periods; ORD-1002 changes from Silver to Gold because the new effective date (2026-08-15) precedes its order date. At INR 83.20 per USD, `42.50 × 83.20 = 3,536.00`; Gold points are `floor(3,536 / 100) × 2 = 70`. Relative to the baseline 126 points, total gold points are predicted to become 161. Exactly one gold row (ORD-1002) is predicted to have a changed row hash and `published_at`. These are predictions; the next cells show actual results only after Snowflake runs.

In [ ]:
before_published = {row[0]: row[1] for row in query(connection, "SELECT order_id, published_at FROM gold.fct_orders")}
run_pipeline(connection, include_arjun_tier_change=True)
after_published = {row[0]: row[1] for row in query(connection, "SELECT order_id, published_at FROM gold.fct_orders")}
changed_published_at = sum(before_published.get(order_id) != published_at for order_id, published_at in after_published.items())
print('Gold rows whose published_at changed:', changed_published_at)
show(connection, "SELECT cust_id, tier, valid_from, valid_to, is_current FROM dim.customers_scd2 WHERE cust_id = 'CUST-02' ORDER BY valid_from")

In [ ]:
show(connection, "SELECT order_id, tier, multiplier, amount_inr, loyalty_points FROM gold.fct_orders WHERE order_id = 'ORD-1002'")
query(connection, "SELECT COUNT(*), SUM(loyalty_points) FROM gold.fct_orders")

`row_hash` covers the business columns used to build each gold row. On a rerun, `MERGE` matches by `order_id` and updates only when the hash differs, refreshing `published_at` only for changed rows. Unchanged gold rows remain untouched. Bronze is an append-only landing layer, so each ordinary run adds 8 order records even though downstream deduplication and the gold merge prevent duplicate business orders. The optional CRM event adds one CRM raw record on its run.

The sixth check counts future-dated rows in `silver.orders_dedup`. It runs in a separate test database so the intentional failure does not contaminate the baseline or Arjun runs. By default the cell uses `FRESHCART_B5_TEST`; set `SNOWFLAKE_B5_DATABASE` in `.env` if you want another isolated database. It uses the same account credentials and creates/selects that database before running. The test-only mock has no amount, a valid customer and pincode, and a future date, so the future-date check should report `PROBLEM` without creating unrelated missing-FX or customer failures. The cleanup below removes the test-only mock from the isolated database after you capture the actual check output.

In [ ]:
qa_connection = connect_snowflake()
qa_database = os.getenv('SNOWFLAKE_B5_DATABASE', 'FRESHCART_B5_TEST')
run_sql(qa_connection, setup_sql(qa_database))
run_pipeline(qa_connection, include_future_order=True)
qa_checks = quality_checks(qa_connection)
qa_checks['future-dated silver orders']

In [ ]:
run_sql(qa_connection, "DELETE FROM bronze.orders_raw WHERE source = 'web' AND raw:order_id::STRING = 'ORD-2099-01'")
run_sql(qa_connection, "DELETE FROM gold.fct_orders WHERE order_id = 'ORD-2099-01'")
run_pipeline(qa_connection)
print('The future test row was removed from the isolated QA database.')

For the written B5 answer, use your lecture notes: explain why retaining raw source records and transforming in the warehouse suits FreshCart's evolving data and traceability needs, then name one lecture case where transforming before loading is preferable. Record actual Snowflake outputs only after running the cells.

In [ ]:
connection.close()
qa_connection.close()

## Recorded evidence: B1–B5

The following results are reproduced from saved run reports and logs. They are **captured evidence**, not outputs from executing this notebook. B1–B5 evidence is in `reports/`; no Snowflake connection was made while preparing this section.

### B1 — baseline snapshot

Source: [`reports/b1_baseline_results.md`](../reports/b1_baseline_results.md). Actual database `FRESHCART_B1`; run ID `20261115_063011_352098`.

| Measure | Recorded result |
|---|---:|
| Bronze raw orders | 8 (6 web, 2 mobile) |
| Clean / quarantined / deduplicated | 6 / 2 / 4 |
| Gold orders | 4 |
| Gold revenue | INR 7,785.00 |
| Gold loyalty points | 126 |
| Rejected share | 25% |

Six recorded quality checks returned `OK`, including the future-date check. This was the one baseline pipeline call; the currency validation rule was already active.

### B2 — record trace

Source: [`reports/b2_record_trace.md`](../reports/b2_record_trace.md), read-only `SELECT` trace after the B1 run (same run ID above).

ORD-1003 appeared in Bronze as two original shapes:

```json
{"order_id":"ORD-1003","cust_email":"meera@outlook.com","cust_name":"MEERA S","order_date":"2026-09-02","amount":null,"currency":"INR","pincode":"400050","updated_at":"2026-09-02 09:10:00"}
```

```json
{"orderId":"ORD-1003","customer":{"email":"Meera@Outlook.com","name":"Meera S"},"orderDate":"02/09/2026","total":{"value":"2,100.00","currency":"INR"},"deliveryPin":"400050","updatedAt":"2026-09-02T18:40:00"}
```

The web fields `order_id/cust_email/cust_name/order_date/amount/currency/pincode/updated_at` and mobile fields `orderId/customer.email/customer.name/orderDate/total.value/total.currency/deliveryPin/updatedAt` map to shared order columns. The amount parser used was:

```sql
TRY_TO_DECIMAL(REGEXP_REPLACE(amount, '[^0-9.-]', ''), 12, 2)
```

This parsed mobile `2,100.00` as `2100.00`; web amount remained NULL. The actual checked rows had no validation reasons. Dedup selected the latest version:

```sql
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY order_id ORDER BY updated_at DESC, ingest_seq DESC
) = 1
```

The 18:40 mobile row (ingest sequence 8) survived over the 09:10 web row (sequence 2). Actual Gold ORD-1003 amount was INR 2,100.00, tier Platinum, multiplier 3; `FLOOR(2100/100) * 3 = 63` points. ORD-1001 (Sept. 1) matched Priya's Silver SCD2 interval: INR 1,250.00 and 12 points. ORD-1005 (Sept. 3) matched Gold: INR 899.00 and 16 points. The trace report includes the full actual rows and queries.

### B3 — currency validation

Source: [`reports/b3_currency_validation.md`](../reports/b3_currency_validation.md). Actual ORD-1006 reasons with the active currency rule:

```json
["bad_email", "bad_pincode", "unsupported_currency"]
```

A read-only SELECT-based **counterfactual** omitting only the currency predicate returned:

```json
["bad_email", "bad_pincode"]
```

The currency rule was already active during B1, so this is not a historical before/after execution. ORD-1006 remains quarantined without that predicate because its malformed email and invalid pincode still make the reasons array non-empty. The report recorded one ORD-1006 quarantine row, two total quarantine rows, and Gold totals of 4 orders / INR 7,785.00 / 126 points at the B3 investigation.

Relevant rule and clean-row condition:

```sql
IFF(currency IS NULL OR currency NOT IN ('INR', 'USD'), 'unsupported_currency', NULL)
FROM silver.orders_checked WHERE ARRAY_SIZE(reasons) = 0;
```

### B4 — prediction versus actual

Sources: [`reports/b4_predictions.md`](../reports/b4_predictions.md) and [`reports/b4_actual_results.md`](../reports/b4_actual_results.md). The actual targeted CRM update run ID was `20261115_070420_455684`; this was not a full pipeline run.

| Measure | Prediction | Recorded actual |
|---|---:|---:|
| CUST-02 SCD2 rows | 2 | 2 |
| ORD-1002 tier | Gold | Gold |
| ORD-1002 points | 70 | 70 |
| Gold total points | 161 | 161 |
| Gold rows with changed `published_at` | 1 | 1 (ORD-1002) |

ORD-1002 changed Silver/multiplier 1/35 points to Gold/multiplier 2/70 points; INR revenue remained INR 7,785.00 overall. The saved actual comparison shows the other three Gold rows retained their row hashes and publication timestamps.

### B5 — future-date quality check

Sources: [`reports/b5_quality_check.md`](../reports/b5_quality_check.md) and the complete [`reports/b5_quality_check_execution.log`](../reports/b5_quality_check_execution.log). The controlled transaction used `FRESHCART_B1`, inserted only temporary ID `B5-MOCK-FUTURE-001` and date `2099-01-01`, and did not call `run_pipeline()`.

```sql
SELECT COUNT(*)
FROM silver.orders_dedup
WHERE order_date > CURRENT_DATE();
```

Recorded targeted result while the temporary row existed:

```text
future-dated silver orders    1   PROBLEM
```

The transaction deleted only the temporary ID. Its count returned to zero; the future-date count returned to zero (`OK`); dedup returned to 4 rows; Gold remained 4 orders, INR 7,785.00 and 161 points. All cleanup checks passed before commit, and post-commit verification found no temporary row.

### Evidence boundary

This notebook section presents saved evidence for review; its code cells remain unexecuted with empty outputs. Each result is attributed to the cited report or log, and no new Snowflake query or write was performed to prepare these cells.

In [1]:
from pathlib import Path
import os
from IPython.display import Markdown, display

def _find_project_root():
    anchors = ("reports/assignment_report.md", "elt/freshcart_elt.ipynb", "iris_project/pipelines/iris_classifier/metadata.yaml")
    starts = []
    configured = os.environ.get("FRESHCART_PROJECT_ROOT")
    if configured:
        starts.append(Path(configured).expanduser())
    if globals().get("__file__"):
        starts.append(Path(__file__).resolve().parent)
    starts.append(Path.cwd().resolve())
    visited = set()
    for start in starts:
        for candidate in (start, *start.parents):
            candidate = candidate.resolve()
            if candidate in visited:
                continue
            visited.add(candidate)
            if all((candidate / anchor).is_file() for anchor in anchors):
                return candidate
    raise FileNotFoundError("Could not locate the project root from FRESHCART_PROJECT_ROOT, the notebook file, or the current directory.")

PROJECT_ROOT = _find_project_root()
REPORTS_DIR = PROJECT_ROOT / "reports"
display(Markdown('## Captured execution evidence (read-only)\n\nThese cells render previously saved reports and logs from this project. They do **not** connect to Snowflake or rerun the pipeline; original pipeline cells above remain unexecuted.'))


## Captured execution evidence (read-only)

These cells render previously saved reports and logs from this project. They do **not** connect to Snowflake or rerun the pipeline; original pipeline cells above remain unexecuted.

In [2]:
freshcart_report_files = [
    "b1_baseline_results.md",
    "b2_record_trace.md",
    "b3_currency_validation.md",
]
for filename in freshcart_report_files:
    evidence_path = REPORTS_DIR / filename
    if not evidence_path.is_file():
        raise FileNotFoundError(evidence_path)
    display(Markdown(f"## Captured evidence: {filename}\n\nSource file: `{evidence_path.relative_to(PROJECT_ROOT).as_posix()}`\n\n---\n\n" + evidence_path.read_text(encoding="utf-8")))


## Captured evidence: b1_baseline_results.md

Source file: `reports/b1_baseline_results.md`

---

# FreshCart B1 Baseline Results

**Database:** `FRESHCART_B1`  
**Role:** `ACCOUNTADMIN`  
**Warehouse:** `COMPUTE_WH`  
**Run ID:** `20261115_063011_352098`

This records the single observed baseline execution. `run_pipeline(connection)` was called once; `baseline_snapshot(connection)` was called immediately afterward. No retry or second pipeline run was made. Credentials are not included.

## Actual pipeline output

```text
web: 6 records -> data\freshcart_landing\web\web__20261115_063011_352098.json.gz
mobile: 2 records -> data\freshcart_landing\mobile\mobile__20261115_063011_352098.json.gz
crm: 4 records -> data\freshcart_landing\crm\crm__20261115_063011_352098.json.gz
fx: 1 records -> data\freshcart_landing\fx\fx__20261115_063011_352098.json.gz
6 rows -> bronze.orders_raw
2 rows -> bronze.orders_raw
4 rows -> bronze.customers_raw
1 rows -> bronze.fx_raw
pincode: 3 records -> data\freshcart_landing\pincode\pincode__20261115_063011_352098.json.gz
3 rows -> bronze.pincode_raw
rows gained or lost while enriching    0   OK
duplicate orders in gold               0   OK
orders with no customer match          0   OK
orders with an amount but no FX rate   0   OK
orders with no city                    0   OK
future-dated silver orders             0   OK
share of rows rejected                 25%
GOLD: 4 orders, revenue INR 7785.00, 126 loyalty points
RUN_ID: 20261115_063011_352098
BASELINE_SNAPSHOT: {'bronze_orders': 8, 'clean_orders': 6, 'quarantined_orders': 2, 'deduplicated_orders': 4, 'gold_orders': 4, 'gold_revenue_inr': Decimal('7785.00'), 'gold_loyalty_points': 126}
rows gained or lost while enriching    0   OK
duplicate orders in gold               0   OK
orders with no customer match          0   OK
orders with an amount but no FX rate   0   OK
orders with no city                    0   OK
future-dated silver orders             0   OK
share of rows rejected                 25%
QUALITY_CHECK_RESULTS: {'rows gained or lost while enriching': 0, 'duplicate orders in gold': 0, 'orders with no customer match': 0, 'orders with an amount but no FX rate': 0, 'orders with no city': 0, 'future-dated silver orders': 0, 'share of rows rejected': 0.25}
```

## Assignment values

| Check | Actual |
| --- | ---: |
| Bronze raw orders | 8 (6 web + 2 mobile) |
| Silver clean orders | 6 |
| Quarantine orders | 2 |
| Deduplicated orders | 4 |
| Gold orders | 4 |
| Gold revenue | INR 7,785.00 |
| Gold loyalty points | 126 |
| Rejected share | 25% |

## Scope note

The existing `run_pipeline()` includes the currency validation rule and six quality checks, including the future-date check; these ran as part of this single baseline call and returned the output shown above. No B3 before/after comparison, Arjun SCD2 event, B5 future-date mock order, isolated B5 run, or follow-up pipeline execution was performed.


## Captured evidence: b2_record_trace.md

Source file: `reports/b2_record_trace.md`

---

# FreshCart B2 Record Trace

**Database:** `FRESHCART_B1`  
**Role:** `ACCOUNTADMIN`  
**Warehouse:** `COMPUTE_WH`  
**Evidence source:** read-only `SELECT` queries executed after the single B1 baseline. No setup or pipeline code was run for this trace.

## Section 1: ORD-1003

### Raw bronze rows

Query used:

```sql
SELECT ingest_seq, source, raw, run_id, extracted_at
FROM FRESHCART_B1.BRONZE.ORDERS_RAW
WHERE raw:order_id::STRING = 'ORD-1003'
   OR raw:orderId::STRING = 'ORD-1003'
ORDER BY ingest_seq;
```

Actual rows:

| ingest_seq | source | run_id | extracted_at |
| ---: | --- | --- | --- |
| 2 | web | `20261115_063011_352098` | `2026-11-15 06:30:11.356583+00:00` |
| 8 | mobile | `20261115_063011_352098` | `2026-11-15 06:30:11.360858+00:00` |

Web raw JSON:

```json
{
  "amount": null,
  "currency": "INR",
  "cust_email": "meera@outlook.com",
  "cust_name": "MEERA S",
  "order_date": "2026-09-02",
  "order_id": "ORD-1003",
  "pincode": "400050",
  "updated_at": "2026-09-02 09:10:00"
}
```

Mobile raw JSON:

```json
{
  "customer": {
    "email": "Meera@Outlook.com",
    "name": "Meera S"
  },
  "deliveryPin": "400050",
  "orderDate": "02/09/2026",
  "orderId": "ORD-1003",
  "total": {
    "currency": "INR",
    "value": "2,100.00"
  },
  "updatedAt": "2026-09-02T18:40:00"
}
```

The web source is a flat object and the mobile source is nested, so their field names differ. The raw bronze records retain those original shapes.

### Unified checked rows

Query used:

```sql
SELECT ingest_seq, source, order_id, email, name, order_date, amount,
       currency, pincode, updated_at, reasons
FROM FRESHCART_B1.SILVER.ORDERS_CHECKED
WHERE order_id = 'ORD-1003'
ORDER BY ingest_seq;
```

Actual rows:

| ingest_seq | source | order_id | email | name | order_date | amount | currency | pincode | updated_at | reasons |
| ---: | --- | --- | --- | --- | --- | ---: | --- | --- | --- | --- |
| 2 | web | ORD-1003 | meera@outlook.com | Meera S | 2026-09-02 | NULL | INR | 400050 | 2026-09-02 09:10:00 | `[]` |
| 8 | mobile | ORD-1003 | meera@outlook.com | Meera S | 2026-09-02 | 2100.00 | INR | 400050 | 2026-09-02 18:40:00 | `[]` |

In `SQL_10_ORDERS_CLEAN`, the web branch maps `raw:order_id`, `raw:cust_email`, `raw:cust_name`, `raw:order_date`, `raw:amount`, `raw:currency`, `raw:pincode`, and `raw:updated_at`. The mobile branch maps `raw:orderId`, `raw:customer.email`, `raw:customer.name`, `raw:orderDate`, `raw:total.value`, `raw:total.currency`, `raw:deliveryPin`, and `raw:updatedAt` to those same columns. The later typed CTE trims/normalizes email, name, currency, and pincode, and parses the dates and timestamp.

The amount expression is `TRY_TO_DECIMAL(REGEXP_REPLACE(amount, '[^0-9.-]', ''), 12, 2)`. For mobile, this removes the comma from `2,100.00` and converts it to `2100.00`. The web amount remains NULL. Both records have an empty `reasons` array.

### Deduplication and Gold

Deduplicated-row query:

```sql
SELECT ingest_seq, source, order_id, email, name, order_date, amount,
       currency, pincode, updated_at, amount_missing
FROM FRESHCART_B1.SILVER.ORDERS_DEDUP
WHERE order_id = 'ORD-1003';
```

Actual surviving row: ingest sequence `8`, source `mobile`, amount `2100.00`, updated at `2026-09-02 18:40:00`, `amount_missing = FALSE`.

The source SQL is:

```sql
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY order_id
  ORDER BY updated_at DESC, ingest_seq DESC
) = 1;
```

The mobile row survives because its `updated_at` is later (18:40 vs. 09:10). The ingest sequence is the tie-breaker if timestamps match.

Gold query:

```sql
SELECT order_id, order_date, cust_id, email, tier, multiplier, amount,
       currency, fx_rate, amount_inr, loyalty_points, is_high_value,
       city, state, region, updated_at, row_hash, published_at
FROM FRESHCART_B1.GOLD.FCT_ORDERS
WHERE order_id = 'ORD-1003';
```

Actual Gold row:

| order_id | order_date | cust_id | email | tier | multiplier | amount | currency | fx_rate | amount_inr | loyalty_points | is_high_value | city | state | region | updated_at | row_hash | published_at |
| --- | --- | --- | --- | --- | ---: | ---: | --- | ---: | ---: | ---: | --- | --- | --- | --- | --- | ---: | --- |
| ORD-1003 | 2026-09-02 | CUST-03 | meera@outlook.com | Platinum | 3 | 2100.00 | INR | 1.0000 | 2100.00 | 63 | TRUE | Mumbai | Maharashtra | West | 2026-09-02 18:40:00 | -1632326326823866187 | 2026-11-14 22:30:28.048000-08:00 |

The Gold SQL computes `amount_inr = ROUND(amount * fx.rate_to_inr, 2)`. The actual rate is `1.0000` for INR, so `2,100.00 × 1.0000 = 2,100.00`. It computes `loyalty_points = FLOOR(amount_inr / 100) * multiplier`, so `FLOOR(2,100 / 100) × 3 = 21 × 3 = 63`.

## Section 2: ORD-1001 and ORD-1005

Gold query used:

```sql
SELECT order_id, order_date, cust_id, email, tier, multiplier, amount,
       currency, fx_rate, amount_inr, loyalty_points, updated_at
FROM FRESHCART_B1.GOLD.FCT_ORDERS
WHERE order_id IN ('ORD-1001', 'ORD-1005')
ORDER BY order_id;
```

Actual rows:

| order_id | order_date | cust_id | email | tier | multiplier | amount | currency | fx_rate | amount_inr | loyalty_points | updated_at |
| --- | --- | --- | --- | --- | ---: | ---: | --- | ---: | ---: | ---: | --- |
| ORD-1001 | 2026-09-01 | CUST-01 | priya.r@gmail.com | Silver | 1 | 1250.00 | INR | 1.0000 | 1250.00 | 12 | 2026-09-01 10:02:00 |
| ORD-1005 | 2026-09-03 | CUST-01 | priya.r@gmail.com | Gold | 2 | 899.00 | INR | 1.0000 | 899.00 | 16 | 2026-09-03 16:45:00 |

SCD2 history query used:

```sql
SELECT cust_id, email, tier, valid_from, valid_to, is_current
FROM FRESHCART_B1.DIM.CUSTOMERS_SCD2
WHERE cust_id = 'CUST-01'
ORDER BY valid_from;
```

Actual history:

| cust_id | email | tier | valid_from | valid_to | is_current |
| --- | --- | --- | --- | --- | --- |
| CUST-01 | priya.r@gmail.com | Silver | 2023-04-10 | 2026-09-02 | FALSE |
| CUST-01 | priya.r@gmail.com | Gold | 2026-09-03 | 9999-12-31 | TRUE |

Multiplier query used:

```sql
SELECT tier, multiplier
FROM FRESHCART_B1.REF.TIER_MULTIPLIER
ORDER BY tier;
```

Actual rows: Gold `2`, Platinum `3`, Silver `1`.

`SQL_40_GOLD` joins `dim.customers_scd2` by email and the point-in-time predicate `o.order_date BETWEEN c.valid_from AND c.valid_to`. ORD-1001 (Sept. 1) falls in Priya's Silver interval, ending Sept. 2. ORD-1005 (Sept. 3) falls in the Gold interval beginning Sept. 3. With INR rate 1.0, points are `FLOOR(1,250 / 100) × 1 = 12` for ORD-1001 and `FLOOR(899 / 100) × 2 = 16` for ORD-1005.

## Execution boundary

All statements listed above were `SELECT` queries. No database objects were modified, no pipeline was run, and no credentials were captured in this file.


## Captured evidence: b3_currency_validation.md

Source file: `reports/b3_currency_validation.md`

---

# FreshCart B3: Unsupported Currency Validation

**Database:** `FRESHCART_B1`  
**Investigation:** Read-only queries against the existing B1 run  
**Baseline run ID:** `20261115_063011_352098`

No pipeline or transformation SQL was executed for this investigation. The evidence below uses the actual ORD-1006 rows and current aggregates from Snowflake.

## Implementation inspected

`SQL_10_ORDERS_CLEAN` normalizes currency with `UPPER(TRIM(currency))`, then includes this validation predicate in `reasons`:

```sql
IFF(currency IS NULL OR currency NOT IN ('INR', 'USD'), 'unsupported_currency', NULL)
```

The rule is present in the implementation and was already active during the B1 run. The clean table admits only rows with no reasons:

```sql
FROM silver.orders_checked WHERE ARRAY_SIZE(reasons) = 0;
```

## ORD-1006 actual reasons

The actual row in `silver.orders_checked` has source `web`, `ingest_seq = 6`, email `vikram@@mail`, pincode `99999`, currency `EUR`, amount `30.00`, and updated time `2026-09-04 08:30:00`.

The actual reasons array, including the active currency rule, is:

```json
["bad_email", "bad_pincode", "unsupported_currency"]
```

The matching row in `quarantine.orders` has the same reasons, source, order ID, ingest sequence, and baseline run ID.

## Counterfactual without the currency rule

A read-only `SELECT` rebuilt the `ARRAY_CONSTRUCT_COMPACT` from the existing `silver.orders_checked` values using all the other validation expressions, but omitted only the currency predicate. Its actual result was:

```json
["bad_email", "bad_pincode"]
```

This is a **SELECT-based counterfactual**, not a result from a historical run with the currency rule disabled. The currency rule was already active during B1, so no historical “before” reasons array was captured.

## Why it remains quarantined

Removing `unsupported_currency` would not make ORD-1006 valid. Its malformed email and invalid five-digit pincode still produce reasons, and `silver.orders_clean` admits only rows where `ARRAY_SIZE(reasons) = 0`. Therefore ORD-1006 remains excluded from the clean/deduplicated path and Gold even without the currency rule.

## Duplicate quarantine prevention

`SQL_10_ORDERS_CLEAN` inserts rejected rows only when the quarantine table does not already contain the same source, null-safe order ID, and updated timestamp:

```sql
AND NOT EXISTS (
  SELECT 1 FROM quarantine.orders q
  WHERE q.source = c.source
    AND EQUAL_NULL(q.order_id, c.order_id)
    AND EQUAL_NULL(q.row_data:UPDATED_AT::TIMESTAMP_NTZ, c.updated_at)
)
```

The actual ORD-1006 quarantine row matches those identifying fields. On a later execution against the same Bronze record, `NOT EXISTS` sees the matching existing row and prevents another quarantine insert for that source/order/version. The separate `QUALIFY ROW_NUMBER()` in the `INSERT` also selects only one candidate per source/order/update timestamp within the current input batch. No reapplication was executed in this investigation.

The current quarantine table contains **2** rows total, including the single observed ORD-1006 row.

## Current Gold totals

At the time of the B3 investigation, the actual read-only aggregate query returned:

| Gold measure | Current value |
| --- | ---: |
| Orders | 4 |
| Revenue | INR 7,785.00 |
| Loyalty points | 126 |

These totals are unaffected by whether ORD-1006 has the additional `unsupported_currency` reason: with or without that reason, `bad_email` and `bad_pincode` remain in its reasons array. Since `silver.orders_clean` requires an empty reasons array, the row stays out of the clean/deduplicated path and Gold. No post-change comparison run was performed.

## Evidence boundary

- Actual with-rule reasons: `["bad_email", "bad_pincode", "unsupported_currency"]`.
- SELECT-based counterfactual without the currency predicate: `["bad_email", "bad_pincode"]`.
- Currency rule was already active during B1; the counterfactual is not historical execution evidence.
- No pipeline, cleaning SQL, deduplication SQL, Gold SQL, or database writes were run for this investigation.


In [3]:
for filename in ("b4_predictions.md", "b4_actual_results.md"):
    evidence_path = REPORTS_DIR / filename
    if not evidence_path.is_file():
        raise FileNotFoundError(evidence_path)
    display(Markdown(f"## Captured evidence: {filename}\n\nSource file: `{evidence_path.relative_to(PROJECT_ROOT).as_posix()}`\n\n---\n\n" + evidence_path.read_text(encoding="utf-8")))


## Captured evidence: b4_predictions.md

Source file: `reports/b4_predictions.md`

---

# FreshCart B4: Pre-change Predictions and Gold Snapshot

**Database:** `FRESHCART_B1`  
**Role:** `ACCOUNTADMIN`  
**Warehouse:** `COMPUTE_WH`  
**Pre-change baseline run ID:** `20261115_063011_352098`

This file records the read-only B4 preparation before any Snowflake write. The proposed event was inspected but has not been added or applied. No setup or pipeline function was run in this investigation.

## Proposed CRM event

```json
{
  "cust_id": "CUST-02",
  "email": "arjun.k@yahoo.in",
  "tier": "Gold",
  "effective_from": "2026-08-15"
}
```

The `bronze.customers_raw` query for CUST-02 found one event, ingest sequence `3`, run ID `20261115_063011_352098`: Silver effective `2024-01-15`. The exact proposed Gold event was not present in the returned Bronze records.

## Actual current SCD2 and ORD-1002

CUST-02 SCD2 row:

| cust_id | email | tier | valid_from | valid_to | is_current |
| --- | --- | --- | --- | --- | --- |
| CUST-02 | arjun.k@yahoo.in | Silver | 2024-01-15 | 9999-12-31 | TRUE |

Current Gold row for ORD-1002:

| order_id | order_date | cust_id | tier | multiplier | amount | currency | fx_rate | amount_inr | loyalty_points | row_hash | published_at |
| --- | --- | --- | --- | ---: | ---: | --- | ---: | ---: | ---: | ---: | --- |
| ORD-1002 | 2026-09-01 | CUST-02 | Silver | 1 | 42.50 | USD | 83.2000 | 3536.00 | 35 | -7698738864412791922 | 2026-11-14 22:30:28.048000-08:00 |

## Actual pre-change Gold snapshot

This exact projection was read before any B4 write:

| order_id | tier | multiplier | loyalty_points | row_hash | published_at |
| --- | --- | ---: | ---: | ---: | --- |
| ORD-1001 | Silver | 1 | 12 | 4284627330879666653 | 2026-11-14 22:30:28.048000-08:00 |
| ORD-1002 | Silver | 1 | 35 | -7698738864412791922 | 2026-11-14 22:30:28.048000-08:00 |
| ORD-1003 | Platinum | 3 | 63 | -1632326326823866187 | 2026-11-14 22:30:28.048000-08:00 |
| ORD-1005 | Gold | 2 | 16 | -231880702048109571 | 2026-11-14 22:30:28.048000-08:00 |

Current totals from Snowflake: **4 orders**, **INR 7,785.00 revenue**, **126 loyalty points**.

## Predictions before execution

| Quantity | Prediction |
| --- | ---: |
| CUST-02 rows in `dim.customers_scd2` after event | 2 |
| Tier for ORD-1002 after event | Gold |
| ORD-1002 loyalty points after event | 70 |
| Total Gold loyalty points after event | 161 |
| Gold rows whose `published_at` changes | 1 |

Reasoning: the event effective `2026-08-15` precedes ORD-1002's `2026-09-01` order date, so the point-in-time SCD2 join should select Gold. The current rate and amount are `USD 42.50 × 83.2000 = INR 3,536.00`. Gold multiplier 2 gives `FLOOR(3,536 / 100) × 2 = 70` points. This changes ORD-1002 from 35 points to 70, so predicted total points are `126 - 35 + 70 = 161`. The row hash includes tier/multiplier and derived points, so ORD-1002 is predicted to be the single changed row and to receive a new `published_at`; all other snapshot rows are predicted to remain unchanged.

These are predictions only. No Gold/SCD2 write, pipeline execution, or post-change verification has occurred.


## Captured evidence: b4_actual_results.md

Source file: `reports/b4_actual_results.md`

---

# FreshCart B4: Actual SCD2 and Gold Results

**Database:** `FRESHCART_B1`  
**Role:** `ACCOUNTADMIN`  
**Warehouse:** `COMPUTE_WH`  
**New CRM run ID:** `20261115_070420_455684`  
**B1 baseline run ID:** `20261115_063011_352098`

This records the approved targeted B4 update. `run_pipeline()` was not called. Only the single new CRM event was written and loaded; then `SQL_30_CUSTOMERS_SCD2` and `SQL_40_GOLD` were each executed once. Credentials are not included.

## Event and execution

Before loading, the exact Gold event was absent from `bronze.customers_raw` (`0` matches). The event written was:

```json
{
  "cust_id": "CUST-02",
  "email": "arjun.k@yahoo.in",
  "tier": "Gold",
  "effective_from": "2026-08-15"
}
```

Actual Bronze load result: one CRM row loaded, `ingest_seq = 5`, run ID `20261115_070420_455684`. No orders, FX, or pincode records were loaded.

## Actual SCD2 history

| cust_id | email | tier | valid_from | valid_to | is_current |
| --- | --- | --- | --- | --- | --- |
| CUST-02 | arjun.k@yahoo.in | Silver | 2024-01-15 | 2026-08-14 | FALSE |
| CUST-02 | arjun.k@yahoo.in | Gold | 2026-08-15 | 9999-12-31 | TRUE |

CUST-02 has exactly 2 rows. The Silver period now ends on 2026-08-14, the day before the Gold event; the Gold period starts on 2026-08-15.

## ORD-1002 Gold result

| order_id | order_date | tier | multiplier | amount | currency | fx_rate | amount_inr | loyalty_points | row_hash | published_at |
| --- | --- | --- | ---: | ---: | --- | ---: | ---: | ---: | ---: | --- |
| ORD-1002 | 2026-09-01 | Gold | 2 | 42.50 | USD | 83.2000 | 3536.00 | 70 | 3392257107350681567 | 2026-11-14 23:04:22.990000-08:00 |

## Gold comparison

The pre-change values below were preserved from `reports/b4_predictions.md` before the database update. Actual post-change values are from a read-only query after `SQL_40_GOLD` completed.

| order_id | Before tier | Before multiplier | Before points | Before row_hash | Before published_at | After tier | After multiplier | After points | After row_hash | After published_at | published_at changed |
| --- | --- | ---: | ---: | ---: | --- | --- | ---: | ---: | ---: | --- | --- |
| ORD-1001 | Silver | 1 | 12 | 4284627330879666653 | 2026-11-14 22:30:28.048000-08:00 | Silver | 1 | 12 | 4284627330879666653 | 2026-11-14 22:30:28.048000-08:00 | No |
| ORD-1002 | Silver | 1 | 35 | -7698738864412791922 | 2026-11-14 22:30:28.048000-08:00 | Gold | 2 | 70 | 3392257107350681567 | 2026-11-14 23:04:22.990000-08:00 | Yes |
| ORD-1003 | Platinum | 3 | 63 | -1632326326823866187 | 2026-11-14 22:30:28.048000-08:00 | Platinum | 3 | 63 | -1632326326823866187 | 2026-11-14 22:30:28.048000-08:00 | No |
| ORD-1005 | Gold | 2 | 16 | -231880702048109571 | 2026-11-14 22:30:28.048000-08:00 | Gold | 2 | 16 | -231880702048109571 | 2026-11-14 22:30:28.048000-08:00 | No |

Exactly one row, ORD-1002, changed `published_at`. No other Gold row changed.

## Gold totals

| Measure | Before | Actual after |
| --- | ---: | ---: |
| Orders | 4 | 4 |
| Revenue | INR 7,785.00 | INR 7,785.00 |
| Loyalty points | 126 | 161 |

The points increase by 35: ORD-1002's INR amount is unchanged at 3,536.00, but the Gold multiplier changes its points from `FLOOR(3,536 / 100) × 1 = 35` to `FLOOR(3,536 / 100) × 2 = 70`. No other orders changed.

## Execution boundary

The targeted execution did not call `run_pipeline()`, reload order/FX/pincode data, or execute B5. Only the single CRM event was loaded, followed by one execution each of `SQL_30_CUSTOMERS_SCD2` and `SQL_40_GOLD`; all verification queries were read-only.


In [4]:
b5_report_path = REPORTS_DIR / "b5_quality_check.md"
b5_log_path = REPORTS_DIR / "b5_quality_check_execution.log"
for evidence_path in (b5_report_path, b5_log_path):
    if not evidence_path.is_file():
        raise FileNotFoundError(evidence_path)
display(Markdown("## Captured evidence: b5_quality_check.md\n\nPreviously recorded B5 report; no database operation is performed by this cell.\n\n---\n\n" + b5_report_path.read_text(encoding="utf-8")))
log_lines = b5_log_path.read_text(encoding="utf-8").splitlines()
start = next((i for i, line in enumerate(log_lines) if line.startswith("CONNECTION=success")), 0)
print("CAPTURED B5 EXECUTION LOG (read from reports/b5_quality_check_execution.log; prior run):")
print("\n".join(log_lines[start:]))


## Captured evidence: b5_quality_check.md

Previously recorded B5 report; no database operation is performed by this cell.

---

# B5 future-date quality check demonstration

## Actual execution summary

The approved controlled test used `FRESHCART_B1` with `ACCOUNTADMIN` and `COMPUTE_WH`. It did not call `run_pipeline()`, reload Bronze, or modify Gold or any source/reference table. One row was inserted into and deleted from `silver.orders_dedup` within a single transaction. The targeted future-date check returned `PROBLEM` while the row existed, then `OK` after cleanup. The transaction committed only after all cleanup and baseline checks passed.

## Before output

| Read-only precondition | Actual value |
|---|---:|
| `B5-MOCK-FUTURE-001` rows | 0 |
| Future-dated `silver.orders_dedup` rows | 0 |
| `silver.orders_dedup` row count | 4 |
| Gold orders | 4 |
| Gold revenue | INR 7,785.00 |
| Gold loyalty points | 161 |

## Actual SQL and check output

The transaction inserted exactly one row, naming only the requested columns:

```sql
BEGIN TRANSACTION;

INSERT INTO FRESHCART_B1.SILVER.ORDERS_DEDUP (ORDER_ID, ORDER_DATE)
VALUES ('B5-MOCK-FUTURE-001', TO_DATE('2099-01-01', 'YYYY-MM-DD'));

SELECT COUNT(*)
FROM FRESHCART_B1.SILVER.ORDERS_DEDUP
WHERE ORDER_DATE > CURRENT_DATE();
```

Actual result displayed using the existing `quality_checks()` output format:

```text
future-dated silver orders             1   PROBLEM
```

Per instruction, only the targeted future-date query ran during the temporary insertion; the full `quality_checks()` function was not called.

## Cleanup and after output

The transaction deleted only the unique temporary ID:

```sql
DELETE FROM FRESHCART_B1.SILVER.ORDERS_DEDUP
WHERE ORDER_ID = 'B5-MOCK-FUTURE-001';
```

Actual cleanup and verification results before commit:

```text
DELETE_ROWS_AFFECTED=1
TEMP_ID_COUNT_AFTER=0
FUTURE_DATE_COUNT_AFTER=0
future-dated silver orders             0   OK
DEDUP_ROW_COUNT_AFTER=4
GOLD_TOTALS_AFTER=orders:4 revenue_inr:7785.00 loyalty_points:161
ALL_PRECOMMIT_CHECKS=passed
COMMIT=success
```

A separate post-commit read-only verification returned:

```text
POST_COMMIT_TEMP_ID_COUNT=0
POST_COMMIT_FUTURE_DATE_COUNT=0
POST_COMMIT_DEDUP_ROW_COUNT=4
POST_COMMIT_GOLD_TOTALS=orders:4 revenue_inr:7785.00 loyalty_points:161
```

## Why this quality gate matters

The check catches orders with dates later than `CURRENT_DATE()` in the deduplicated Silver table. Such records can distort time-based reporting or appear in future periods before they should. This controlled row demonstrated the selected check's failure condition without re-running transformations or changing Gold or Bronze.

The targeted check detected one future-dated row and returned to zero after cleanup. No temporary record remains. If any step had failed before commit, the transaction handler would roll back and check that the temporary ID was absent; no retry was attempted.

## Complete actual output

The complete captured execution output, including exact SQL statements, preconditions, in-transaction results, commit, post-commit verification, and connector warnings, is saved in [b5_quality_check_execution.log](b5_quality_check_execution.log). A scan found no credentials in the log.



CAPTURED B5 EXECUTION LOG (read from reports/b5_quality_check_execution.log; prior run):
CONNECTION=success role=ACCOUNTADMIN warehouse=COMPUTE_WH database=FRESHCART_B1
=== BEFORE TRANSACTION: read-only preconditions ===
SQL> SELECT COUNT(*) FROM FRESHCART_B1.SILVER.ORDERS_DEDUP WHERE ORDER_ID = 'B5-MOCK-FUTURE-001'
TEMP_ID_COUNT_BEFORE=0
SQL> SELECT COUNT(*) FROM FRESHCART_B1.SILVER.ORDERS_DEDUP WHERE ORDER_DATE > CURRENT_DATE()
FUTURE_DATE_COUNT_BEFORE=0
SQL> SELECT COUNT(*) FROM FRESHCART_B1.SILVER.ORDERS_DEDUP
DEDUP_ROW_COUNT_BEFORE=4
SQL> SELECT COUNT(*), SUM(AMOUNT_INR), SUM(LOYALTY_POINTS) FROM FRESHCART_B1.GOLD.FCT_ORDERS
GOLD_TOTALS_BEFORE=orders:4 revenue_inr:7785.00 loyalty_points:161
SQL> BEGIN TRANSACTION;
SQL> INSERT INTO FRESHCART_B1.SILVER.ORDERS_DEDUP (ORDER_ID, ORDER_DATE) VALUES ('B5-MOCK-FUTURE-001', TO_DATE('2099-01-01', 'YYYY-MM-DD'));
INSERT_ROWS_AFFECTED=1
=== TARGETED FUTURE-DATE QUALITY CHECK WHILE MOCK EXISTS ===
SQL> SELECT COUNT(*) FROM FRESHCART_B1.SILVER.